In [1]:
!pip install -q sentence-transformers huggingface_hub
!pip install -q llama-cpp-python

#this cell took ~9.17 mins to be executed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 MB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00


In [18]:
import json


with open("/content/qa_dataset.json") as f:
    data = json.load(f)

records = []
for qa in data:
    records.append({
        "instruction": qa["question"],
        "output": qa["answer"],
        "module": qa["module"],
    })

len(records)

822

In [14]:

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

instructions = [r["instruction"] for r in records]
instruction_embeddings = embedder.encode(instructions, normalize_embeddings=True, show_progress_bar=True)
instruction_embeddings = np.array(instruction_embeddings)

module_indices = {}
for i, r in enumerate(records):
    module_indices.setdefault(r["module"], []).append(i)

available_modules = sorted(module_indices.keys())
print("Available modules:", available_modules)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Available modules: ['Damaged Building Detection', 'Data Acquisition', 'Data Annotation', 'Training and Preprocessing']


In [15]:

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
    filename="qwen2.5-1.5b-instruct-q4_k_m.gguf",
)

llm = Llama(model_path=model_path, n_ctx=2048, n_threads=4, chat_format="chatml", verbose=False)

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [16]:

SIMILARITY_THRESHOLD = 0.55
FALLBACK_MESSAGE = "That question isn't covered in the flood detection project knowledge base."

REPHRASE_SYSTEM_PROMPT = (
    "You rephrase technical answers so they sound more conversational. "
    "Keep every fact, number, threshold, unit, and formula exactly as given. "
    "Do not add or remove information. Only adjust sentence structure and tone."
)

# cache rephrased answers by record index, so the same matched record
rephrase_cache = {}

def rephrase(answer_text, record_idx):
    if record_idx in rephrase_cache:
        return rephrase_cache[record_idx]

    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": REPHRASE_SYSTEM_PROMPT},
            {"role": "user", "content": f"Rephrase this answer: {answer_text}"},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    result = response["choices"][0]["message"]["content"].strip()
    rephrase_cache[record_idx] = result
    return result

def answer_question(question, module=None):
    """
    module: one of available_modules. If given, search is restricted to
    that module's records only, which removes cross-module ambiguity and
    speeds up the search.
    """
    if module is not None:
        if module not in module_indices:
            return f"Unknown module: {module}", None, None
        candidate_idx = module_indices[module]
    else:
        candidate_idx = list(range(len(records)))

    query_embedding = embedder.encode(question, normalize_embeddings=True)
    candidate_embeddings = instruction_embeddings[candidate_idx]
    scores = candidate_embeddings @ query_embedding

    best_local = int(np.argmax(scores))
    best_score = float(scores[best_local])
    best_idx = candidate_idx[best_local]

    if best_score < SIMILARITY_THRESHOLD:
        return FALLBACK_MESSAGE, None, best_score

    record = records[best_idx]
    final_answer = rephrase(record["output"], best_idx)
    return final_answer, record["module"], best_score

In [17]:

test_cases = [
    ("How do I draw an area of interest for downloading satellite imagery?", "Data Acquisition"),
    ("What annotation tools are used to label flood images?", "Data Annotation"),
    ("What learning rate and batch size are used during training?", "Training and Preprocessing"),
    ("How does the model identify damaged buildings after a flood?", "Damaged Building Detection"),
    ("What's the best recipe for a chocolate cake?", "Data Acquisition"),  # should fall back even with a module set
    ("What is MUST-Former and what problem does it solve?", "Training and Preprocessing"),
    ("What chip size and resolution does the pipeline export?", "Data Acquisition"),
]

for q, module in test_cases:
    answer, matched_module, score = answer_question(q, module=module)
    print(f"Module: {module}")
    print(q)
    print(f"matched_module: {matched_module}, score: {score}")
    print(answer)
    print()


Module: Data Acquisition
How do I draw an area of interest for downloading satellite imagery?
matched_module: Data Acquisition, score: 0.9471491575241089
Draw a shape on the map using the drawing tool. This will be your target area for downloading satellite data.

Module: Data Annotation
What annotation tools are used to label flood images?
matched_module: Data Annotation, score: 0.949933648109436
These tools are used to automatically label flood images: Lee speckle filter, Kittler-Illingworth thresholding, MNDWI water index, AWEI water index, and SAR-optical fusion.

Module: Training and Preprocessing
What learning rate and batch size are used during training?
matched_module: Training and Preprocessing, score: 0.9503002166748047
The learning rate is 0.00005. For SAR, Optical, Projector, and PCA, the batch size is 8. CrossAttn uses a batch size of 4.

Module: Damaged Building Detection
How does the model identify damaged buildings after a flood?
matched_module: Damaged Building Detecti